# Week 3 Pipeline: AI Incident and Risk Explorer
**Topic 3 Lab: Data Engineering for AI**

**Pathway used:** B (CSV / Excel), run twice (one run per source):
- **S01** AI Incident Database (AIID) `incidents.csv` from a weekly snapshot, which supports FR1, FR2, FR4 and RAI1.
- **S02** MIT AI Risk Repository spreadsheet, which supports FR3 and RAI1.

In [43]:
# Cell 1 — Workspace setup
from pathlib import Path

# In Colab the standard path is used. The fallback is only for testing outside Colab.
_IN_COLAB = Path("/content").exists()
BASE_DIR = Path("/content/week3_project") if _IN_COLAB else Path.cwd() / "week3_project"
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
DOCS_DIR = BASE_DIR / "docs"
for d in (RAW_DIR, PROCESSED_DIR, DOCS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Compatibility aliases used by the lab examples
RAW, BASE, OUT = RAW_DIR, BASE_DIR, PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


## Part A — Start with the Data

### Step A1: From Topic 2 requirements to data needs

| Requirement ID | Needed information / feature | Candidate source | Why usable / constraint |
|---|---|---|---|
| FR1, FR2: answer questions and find similar incidents | Incident ID, title, description, date | S01: AIID `incidents.csv` (weekly snapshot) | Authoritative public incident catalogue. Constraint: reporting bias towards widely covered, English-language cases. |
| FR4 / RAI1: cite a verifiable record for every claim | Stable record ID and a source link | S01 (incident ID, link built from the ID) and S02 (risk record ID) | IDs must be unique and non-missing so citations can be checked. |
| FR3: classify into risk domains | Risk domain, sub-domain, risk description | S02: MIT AI Risk Repository | Defines the shared taxonomy the classifier will use. Constraint: licence for redistribution must be confirmed. |
| NFR5: refresh without manual steps | Snapshot date, file fingerprint | S01 and S02 manifests | Repeatable script and SHA-256 fingerprints show when the source changed. |

### Step A2: Config and workspace check

In [44]:
# CONFIG CELL
DATA_MODE = "real"   # "real" = your AIID + MIT files | "synthetic" = labelled practice data only

AIID_SNAPSHOT_DATE = "[2026-11-02]"
RISK_REPO_VERSION = "[Version 4, updated 03 December 2025]"

INCIDENTS_REL_PATH = None   # e.g. "aiid_snapshot/incidents.csv" (path under data/raw)
RISKS_REL_PATH = None       # e.g. "ai_risk_repository.xlsx"
RISK_SHEET = None           # exact sheet name if the wrong sheet is detected

INCLUDE_PREPARED_DATA_IN_ZIP = True

assert DATA_MODE in ("real", "synthetic"), "DATA_MODE must be 'real' or 'synthetic'"
print("DATA_MODE:", DATA_MODE)

DATA_MODE: real


In [45]:
# Cell 2a — Practice data (runs only when DATA_MODE == "synthetic")
import numpy as np
import pandas as pd

if DATA_MODE == "synthetic":
    from openpyxl import Workbook
    rng = np.random.default_rng(42)
    n = 60
    inc = pd.DataFrame({
        "incident_id": [str(i) for i in range(1, n + 1)],
        "date": pd.date_range("2019-01-01", periods=n, freq="30D").strftime("%Y-%m-%d"),
        "title": [f"[SYNTHETIC] Practice incident {i}" for i in range(1, n + 1)],
        "description": [f"Synthetic practice description number {i}." for i in range(1, n + 1)],
        "Alleged deployer of AI system": rng.choice(["Org A", "Org B", "Org C"], n),
    })
    inc.loc[3, "title"] = "   [SYNTHETIC] Practice incident 4   "   # whitespace problem
    inc.loc[7, "description"] = None                              # missing description
    inc.loc[11, "description"] = "   "                            # blank description
    inc = pd.concat([inc, inc.iloc[[2, 5, 9]]], ignore_index=True)  # 3 exact duplicates
    inc.loc[len(inc)] = [None, "2024-01-01", "[SYNTHETIC] row without id", "no id", "Org A"]
    inc.to_csv(RAW_DIR / "incidents.csv", index=False)

    domains = ["1. Discrimination & toxicity", "2. Privacy & security", "3. Misinformation",
               "4. Malicious actors & misuse", "5. Human-computer interaction",
               "6. Socioeconomic & environmental", "7. AI system safety, failures & limitations"]
    risks = [[f"R{i:03d}", f"Practice risk category {i % 5}", f"Practice subcategory {i % 9}",
              f"Synthetic practice risk description {i}.", domains[i % 7], f"Sub-domain {i % 7}.{i % 3}",
              rng.choice(["Human", "AI"]), rng.choice(["Intentional", "Unintentional"]),
              rng.choice(["Pre-deployment", "Post-deployment"])] for i in range(1, 41)]
    risks.append(risks[4])  # one exact duplicate
    wb = Workbook(); ws = wb.active; ws.title = "AI Risk Database"
    ws.append(["SYNTHETIC stand-in for the AI Risk Repository (practice only)"])
    ws.append(["The header row follows after a title row, as in many real spreadsheets."])
    ws.append(["Ev_ID", "Risk category", "Risk subcategory", "Description", "Domain", "Sub-domain",
               "Entity", "Intent", "Timing"])
    for r in risks:
        ws.append(r)
    wb.save(RAW_DIR / "ai_risk_repository_synthetic.xlsx")
    print("Synthetic practice files written to", RAW_DIR)
else:
    print("DATA_MODE is 'real': no practice data created.")

DATA_MODE is 'real': no practice data created.


In [46]:
# Cell 2b — Extract incidents.csv from an AIID snapshot archive
import tarfile

archives = sorted(RAW_DIR.glob("*.tar.bz2"))
if not archives:
    print("No .tar.bz2 archive found. Upload incidents.csv directly or skip this cell.")
else:
    target_dir = RAW_DIR / "aiid_snapshot"
    target_dir.mkdir(exist_ok=True)
    found = False
    with tarfile.open(archives[0], "r:bz2") as tar:
        for member in tar:
            if member.isfile() and Path(member.name).name == "incidents.csv":
                with tar.extractfile(member) as src, (target_dir / "incidents.csv").open("wb") as dst:
                    dst.write(src.read())
                print("Extracted:", member.name, "->", target_dir / "incidents.csv")
                found = True
                break
    if not found:
        print("incidents.csv was not found inside", archives[0].name,
              "- check the archive contents or upload the CSV directly.")

No .tar.bz2 archive found. Upload incidents.csv directly or skip this cell.


In [47]:
# Cell 3 — Verify uploaded inputs
from pathlib import Path
RAW = Path(RAW_DIR)
files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)
assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 2
The AI Risk Repository V4_03_12_2025.xlsx bytes: 712682
incidents.csv bytes: 1336750
Upload verified. Select your relevant input(s) for Step A3.


In [48]:
# Cell 4 — Source selection and lineage utilities
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records,
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

print("Cell 4 completed successfully.")
print("Available functions: select_source(), source_hash(), save_manifest()")

Cell 4 completed successfully.
Available functions: select_source(), source_hash(), save_manifest()


In [49]:
# Cell 4A — Common lineage registry
from pathlib import Path

lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    key = (str(source), str(output))
    lineage_entries[key] = {"source": source, "output": output, "processing": processing}
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


### Step A3: Identify the data type and inspect two sources

**Data type:** both sources are **structured tables** (CSV / Excel), so Pathway B applies. The two sources are two separate files, which satisfies the "inspect two sources" rule. The helper functions below are shared by Part B and Part C so both parts use identical logic.

In [50]:
# Cell A3 — Inspect both sources (shared helpers + inspection evidence)
import re
import pandas as pd

def norm(s):
    return re.sub(r"[\s_]+", " ", str(s).strip().lower())

# Column specs: canonical name -> accepted source column names (normalised: lower case, spaces for _)
INCIDENT_SPEC = {
    "record_id": ["incident id", "incidentid", "id"],
    "title": ["title", "incident title"],
    "description": ["description", "incident description"],
    "date": ["date", "incident date"],
}
INCIDENT_REQUIRED = ["record_id", "title", "description"]
INCIDENT_EXTRA_KEYWORDS = ["deployer", "developer", "harmed"]

RISK_SPEC = {
    "record_id": ["ev id", "evid", "evidence id", "id"],
    "risk_category": ["risk category", "category"],
    "risk_subcategory": ["risk subcategory", "subcategory"],
    "description": ["description"],
    "domain": ["domain"],
    "subdomain": ["sub domain", "subdomain", "sub-domain"],
}
RISK_REQUIRED = ["record_id", "description", "domain"]
RISK_EXTRA_KEYWORDS = ["entity", "intent", "timing"]

def resolve_columns(df, spec, label, required):
    lookup = {norm(c): c for c in df.columns}
    mapping = {}
    for canon, candidates in spec.items():
        for cand in candidates:
            if cand in lookup:
                mapping[canon] = lookup[cand]
                break
    missing = [k for k in required if k not in mapping]
    if missing:
        raise ValueError(
            f"[{label}] Could not find column(s) for {missing}. "
            f"Available columns: {list(df.columns)}. "
            "Edit the *_SPEC lists in this cell or check the source file."
        )
    return mapping

def extra_columns(df, mapping, keywords):
    used = set(mapping.values())
    return [c for c in df.columns if c not in used and any(k in norm(c) for k in keywords)]

def load_incidents(path):
    return pd.read_csv(path, dtype="string", encoding="utf-8")

def load_risks(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, dtype="string", encoding="utf-8")
    sheets = [RISK_SHEET] if RISK_SHEET else pd.ExcelFile(path).sheet_names
    for sheet in sheets:
        probe = pd.read_excel(path, sheet_name=sheet, header=None, nrows=15, dtype="string")
        for row_idx in range(len(probe)):
            cells = {norm(v) for v in probe.iloc[row_idx].dropna()}
            has_id = any(c in cells for c in RISK_SPEC["record_id"])
            if "description" in cells and "domain" in cells and has_id:
                return pd.read_excel(path, sheet_name=sheet, header=row_idx, dtype="string")
    raise ValueError(
        f"No sheet with a header row containing 'Description', 'Domain' and an ID column was found. "
        f"Sheets: {pd.ExcelFile(path).sheet_names}. Set RISK_SHEET in the config cell."
    )

def short(text, n=90):
    text = "" if pd.isna(text) else str(text).strip()
    return text if len(text) <= n else text[: n - 3] + "..."

def inspect_table(df, mapping, label):
    rid, desc = mapping["record_id"], mapping["description"]
    d = df[desc].astype("string")
    longest = d.fillna("").str.len().idxmax()
    reps = []
    for idx in sorted({df.index[0], longest}):
        row = df.loc[idx]
        title_col = mapping.get("title") or mapping.get("risk_category")
        reps.append({"row_index": int(idx), "record_id": short(row[rid], 40),
                     "label": short(row[title_col], 90) if title_col else ""})
    evidence = {
        "rows": int(len(df)),
        "columns": [str(c) for c in df.columns],
        "column_count": int(df.shape[1]),
        "mapped_columns": {k: str(v) for k, v in mapping.items()},
        "missing_mapped": {k: int(df[v].isna().sum()) for k, v in mapping.items()},
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "duplicate_ids": int(df[rid].dropna().duplicated().sum()),
        "representative_records": reps,
    }
    if "date" in mapping:
        parsed = pd.to_datetime(df[mapping["date"]], errors="coerce")
        evidence["unparseable_dates"] = int(parsed.isna().sum() - df[mapping["date"]].isna().sum())
    print(f"--- Inspection: {label} ---")
    print("Rows/columns:", df.shape)
    print("Columns:", list(df.columns))
    print("Mapped fields:", evidence["mapped_columns"])
    print("Missing (mapped fields):", evidence["missing_mapped"])
    print("Exact duplicate rows:", evidence["exact_duplicate_rows"], "| duplicate IDs:", evidence["duplicate_ids"])
    print("Two representative records:", reps)
    return evidence

# --- S01: AIID incidents ---
f_s01 = select_source(relative_path=INCIDENTS_REL_PATH) if INCIDENTS_REL_PATH else select_source(filename="incidents.csv")
df_s01 = load_incidents(f_s01)
assert not df_s01.empty, "S01 is empty."
map_s01 = resolve_columns(df_s01, INCIDENT_SPEC, "S01 incidents", INCIDENT_REQUIRED)
extras_s01 = extra_columns(df_s01, map_s01, INCIDENT_EXTRA_KEYWORDS)

# --- S02: MIT AI Risk Repository ---
f_s02 = select_source(relative_path=RISKS_REL_PATH) if RISKS_REL_PATH else select_source(suffixes={".xlsx"})
df_s02 = load_risks(f_s02)
assert not df_s02.empty, "S02 is empty."
map_s02 = resolve_columns(df_s02, RISK_SPEC, "S02 risk repository", RISK_REQUIRED)
extras_s02 = extra_columns(df_s02, map_s02, RISK_EXTRA_KEYWORDS)

EVIDENCE = {
    "S01": inspect_table(df_s01, map_s01, "S01 AIID incidents"),
    "S02": inspect_table(df_s02, map_s02, "S02 MIT AI Risk Repository"),
}
for key, path in (("S01", f_s01), ("S02", f_s02)):
    EVIDENCE[key]["relative_path"] = str(path.relative_to(RAW_DIR))
    EVIDENCE[key]["bytes"] = int(path.stat().st_size)
    EVIDENCE[key]["sha256"] = source_hash(path)
(PROCESSED_DIR / "a3_inspection_evidence.json").write_text(json.dumps(EVIDENCE, indent=2), encoding="utf-8")
print("\nSaved:", PROCESSED_DIR / "a3_inspection_evidence.json")

Candidates: ['incidents.csv']
Candidates: ['The AI Risk Repository V4_03_12_2025.xlsx']
--- Inspection: S01 AIID incidents ---
Rows/columns: (1786, 9)
Columns: ['_id', 'incident_id', 'date', 'reports', 'Alleged deployer of AI system', 'Alleged developer of AI system', 'Alleged harmed or nearly harmed parties', 'description', 'title']
Mapped fields: {'record_id': 'incident_id', 'title': 'title', 'description': 'description', 'date': 'date'}
Missing (mapped fields): {'record_id': 0, 'title': 0, 'description': 0, 'date': 0}
Exact duplicate rows: 0 | duplicate IDs: 0
Two representative records: [{'row_index': 0, 'record_id': '23', 'label': 'Las Vegas Self-Driving Bus Involved in Accident'}, {'row_index': 1254, 'record_id': '1266', 'label': 'Purportedly AI-Generated Sexual Images of At Least 400 Minors at Zacatecas School Were ...'}]
--- Inspection: S02 MIT AI Risk Repository ---
Rows/columns: (2574, 19)
Columns: ['Title', 'QuickRef', 'Ev_ID', 'Paper_ID', 'Cat_ID', 'SubCat_ID', 'AddEv_ID', 

### Step A4: Data inventory checkpoint
## Part B — Prepare the Data

### Step B1: Pathway selection
**Pathway B (CSV/Excel)** applies to both sources. Pathways A, C, D, E and F are not used. The transformation is deliberately small and does not impute missing values or change the meaning of any text.

| Decision | Choice | Reason (Topic 2 link) |
|---|---|---|
| Standardise column names | `record_id`, `title`, `description`, `date`, … | One schema for later retrieval and citation (FR1, FR4) |
| Trim whitespace; blank text becomes missing | Yes | Clean joins and correct missing counts |
| Drop rows with no ID | Yes | A record without an ID cannot be cited (FR4, RAI1) |
| Remove exact duplicate rows | Yes | Avoids duplicate retrieval results (FR2) |
| Build the incident link from the ID | Yes (S01 only) | FR4 needs a source link; pattern `https://incidentdatabase.ai/cite/<id>` |
| Fill or delete rows with missing descriptions | **No**, flag only | Quality review belongs to Topic 4 |

In [51]:
# Shared preparation function (used by Part B and Part C)
def prepare_table_df(df, mapping, extras, cfg):
    cols = list(mapping.values()) + list(extras)
    out = df[cols].copy().rename(columns={v: k for k, v in mapping.items()})
    stats = {"rows_in": int(len(out))}
    for c in out.columns:
        out[c] = out[c].astype("string")
    if cfg["trim_text"]:
        for c in out.columns:
            out[c] = out[c].str.strip()
            blank = out[c].fillna("NOT_BLANK").eq("")
            out.loc[blank, c] = pd.NA
    before = len(out)
    out = out[out["record_id"].notna()].copy()          # records without ID cannot be cited
    stats["rows_dropped_missing_id"] = int(before - len(out))
    out["record_id"] = out["record_id"].str.replace(r"\.0$", "", regex=True)
    before = len(out)
    if cfg["remove_exact_duplicates"]:
        out = out.drop_duplicates().copy()
    stats["exact_duplicates_removed"] = int(before - len(out))
    if cfg.get("source_url_template"):
        out["source_url"] = out["record_id"].map(lambda x: cfg["source_url_template"].format(record_id=x))
    stats["rows_out"] = int(len(out))
    stats["missing_description_flagged"] = int(out["description"].isna().sum())
    stats["duplicate_ids_remaining"] = int(out["record_id"].duplicated().sum())
    return out.reset_index(drop=True), stats

CONFIG_S01 = {"remove_exact_duplicates": True, "trim_text": True,
              "source_url_template": "https://incidentdatabase.ai/cite/{record_id}"}
CONFIG_S02 = {"remove_exact_duplicates": True, "trim_text": True, "source_url_template": None}

def describe_config(cfg):
    return (f"standardise columns; trim_text={cfg['trim_text']}; drop rows without record_id; "
            f"remove_exact_duplicates={cfg['remove_exact_duplicates']}; "
            f"source_url_template={cfg['source_url_template']}; missing descriptions flagged not imputed")

STATS = {}
print("Preparation helpers ready.")

Preparation helpers ready.


In [52]:
# Cell 5B-S01 — Pathway B for S01 (AIID incidents): prepare, export, validate, register lineage
prepared_s01, STATS["S01"] = prepare_table_df(df_s01, map_s01, extras_s01, CONFIG_S01)
assert not prepared_s01.empty, "No prepared S01 rows remain."

prepared_path_s01 = PROCESSED_DIR / "prepared_incidents.csv"
prepared_s01.to_csv(prepared_path_s01, index=False)

# B3 basic output contract + read-back check
assert prepared_path_s01.exists(), "Prepared output missing."
assert prepared_path_s01.stat().st_size > 0, "Prepared output is empty."
exported = pd.read_csv(prepared_path_s01, dtype="string")
assert len(exported) == len(prepared_s01), "Exported row count does not match."
assert exported.columns.tolist() == prepared_s01.columns.tolist(), "Exported columns do not match."
assert exported["record_id"].notna().all(), "Prepared output contains a record without an ID."
print("Output:", prepared_path_s01, "Size:", prepared_path_s01.stat().st_size)
print("Stats:", STATS["S01"])
print("Prepared sample:")
print(prepared_s01.head(3).to_string(index=False, max_colwidth=60))
print("Validation: PASS (file exists, non-empty, read-back matches; inspect the sample above too)")

# B4.1 lineage registration (S01)
register_lineage(source=f_s01, output=prepared_path_s01, processing=describe_config(CONFIG_S01))

Output: /content/week3_project/data/processed/prepared_incidents.csv Size: 1301519
Stats: {'rows_in': 1786, 'rows_dropped_missing_id': 0, 'exact_duplicates_removed': 0, 'rows_out': 1786, 'missing_description_flagged': 0, 'duplicate_ids_remaining': 0}
Prepared sample:
record_id                                                    title                                                  description       date    Alleged deployer of AI system   Alleged developer of AI system           Alleged harmed or nearly harmed parties                          source_url
       23          Las Vegas Self-Driving Bus Involved in Accident A self-driving public shuttle by Keolis North America and... 2017-11-08 ["navya","keolis-north-america"] ["navya","keolis-north-america"] ["navya","keolis-north-america","bus-passengers"] https://incidentdatabase.ai/cite/23
        4                     Uber AV Killed Pedestrian in Arizona An Uber autonomous vehicle (AV) in autonomous mode struck... 2018-03-18            

In [53]:
# Cell 5B-S02 — Pathway B for S02 (MIT AI Risk Repository): prepare, export, validate, register lineage
prepared_s02, STATS["S02"] = prepare_table_df(df_s02, map_s02, extras_s02, CONFIG_S02)
assert not prepared_s02.empty, "No prepared S02 rows remain."

prepared_path_s02 = PROCESSED_DIR / "prepared_risks.csv"
prepared_s02.to_csv(prepared_path_s02, index=False)

assert prepared_path_s02.exists(), "Prepared output missing."
assert prepared_path_s02.stat().st_size > 0, "Prepared output is empty."
exported = pd.read_csv(prepared_path_s02, dtype="string")
assert len(exported) == len(prepared_s02), "Exported row count does not match."
assert exported.columns.tolist() == prepared_s02.columns.tolist(), "Exported columns do not match."
assert exported["record_id"].notna().all(), "Prepared output contains a record without an ID."
print("Output:", prepared_path_s02, "Size:", prepared_path_s02.stat().st_size)
print("Stats:", STATS["S02"])
print("Domain values found:", sorted(prepared_s02["domain"].dropna().unique())[:12])
print("Prepared sample:")
print(prepared_s02.head(3).to_string(index=False, max_colwidth=50))
print("Validation: PASS (file exists, non-empty, read-back matches; inspect the sample above too)")

register_lineage(source=f_s02, output=prepared_path_s02, processing=describe_config(CONFIG_S02))

Output: /content/week3_project/data/processed/prepared_risks.csv Size: 969797
Stats: {'rows_in': 2574, 'rows_dropped_missing_id': 0, 'exact_duplicates_removed': 0, 'rows_out': 2574, 'missing_description_flagged': 926, 'duplicate_ids_remaining': 5}
Domain values found: ['1. Discrimination & Toxicity', '2. Privacy & Security', '3. Misinformation', '4. Malicious Actors & Misuse', '5. Human-Computer Interaction', '6. Socioeconomic and Environmental', '7. AI System Safety, Failures, & Limitations']
Prepared sample:
 record_id                       risk_category risk_subcategory                                        description                             domain                subdomain Entity            Intent    Timing
  01.00.00                                <NA>             <NA>                                               <NA>                               <NA>                     <NA>   <NA>              <NA>      <NA>
  01.01.00 Type 1: Diffusion of responsibility             <NA> 

### Step B3: Compare original data with the prepared configuration

In [54]:
# Cell B3 — Compare configuration A (as loaded) with configuration B (prepared)
def table_profile(df, id_col, desc_col):
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    edge_ws = 0
    for c in text_cols:
        s = df[c].dropna().astype("string")
        edge_ws += int((s != s.str.strip()).sum())
    desc = df[desc_col].astype("string")
    blank_desc = desc.isna() | desc.fillna("").str.strip().eq("")
    return {"rows": int(len(df)),
            "exact_duplicate_rows": int(df.duplicated().sum()),
            "rows_missing_id": int(df[id_col].isna().sum()),
            "missing_or_blank_description": int(blank_desc.sum()),
            "cells_with_edge_whitespace": edge_ws}

COMPARISON = {
    "S01": {"A_as_loaded": table_profile(df_s01, map_s01["record_id"], map_s01["description"]),
            "B_prepared": table_profile(prepared_s01, "record_id", "description")},
    "S02": {"A_as_loaded": table_profile(df_s02, map_s02["record_id"], map_s02["description"]),
            "B_prepared": table_profile(prepared_s02, "record_id", "description")},
}
for key in COMPARISON:
    print(f"--- {key} ---")
    print(pd.DataFrame(COMPARISON[key]).to_string())
(PROCESSED_DIR / "b3_comparison.json").write_text(json.dumps(COMPARISON, indent=2), encoding="utf-8")
print("\nSaved:", PROCESSED_DIR / "b3_comparison.json")

--- S01 ---
                              A_as_loaded  B_prepared
rows                                 1786        1786
exact_duplicate_rows                    0           0
rows_missing_id                         0           0
missing_or_blank_description            0           0
cells_with_edge_whitespace             92           0
--- S02 ---
                              A_as_loaded  B_prepared
rows                                 2574        2574
exact_duplicate_rows                    0           0
rows_missing_id                         0           0
missing_or_blank_description          926         926
cells_with_edge_whitespace           2765           0

Saved: /content/week3_project/data/processed/b3_comparison.json


### Step B4: Common lineage manifest

In [55]:
# Cell 6A — Generate common lineage manifest
assert lineage_entries, "No lineage entries registered. Register at least one selected pathway first."
lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    source = entry["source"]; output = entry["output"]; processing = entry["processing"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"
    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending",
    })
manifest = save_manifest(lineage_records)
print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print("Saved to:", BASE_DIR / "data" / "week3_manifest.json")

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 2
Saved to: /content/week3_project/data/week3_manifest.json


In [56]:
# Cell 6B — Verify common lineage manifest
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), "Common lineage manifest not found."
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, "No lineage records found in saved manifest."
assert len(saved_records) == len(lineage_entries), "Saved lineage count does not match registration."
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], f"Source checksum mismatch: {source}"
    assert source_hash(output) == record["prepared_sha256"], f"Prepared output checksum mismatch: {output}"
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 2
Manifest: /content/week3_project/data/week3_manifest.json


## Part C — Build and Document the V1 Data Pipeline

### Step C1: Integrated V1 pipeline

In [57]:
# Cell 7 — Common V1 pipeline utilities
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_RUNS = {}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(f"Source folder not found: {source}")
    else:
        if not source.is_file():
            raise FileNotFoundError(f"Source file not found: {source}")
    return source

def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    source = Path(source)
    # INGEST
    if kind == "images":
        inputs = sorted(p for p in source.iterdir()
                        if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"})
    else:
        inputs = [source]
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")
    # INSPECT
    inspection = inspect_fn(inputs, configuration)
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)
    # PREPARE / STORE
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]
    # VALIDATE
    if not outputs:
        raise ValueError("No prepared outputs were produced.")
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0:
            raise ValueError(f"Prepared output is empty: {output}")
    validation = {"output_count": len(outputs),
                  "output_sha256": [sha256_file(p) for p in outputs],
                  "output_bytes": [p.stat().st_size for p in outputs]}
    # RECORD LINEAGE
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [{"path": str(p), "sha256": sha256_file(p)} for p in inputs],
        "outputs": [str(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": "Not verified — check before sharing",
    }
    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    print("Manifest:", manifest_path)
    return record

def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first

print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).


In [58]:
# Cell 8B — Pathway B pipeline runs for S01 and S02

SOURCE_S01 = f_s01          # explicit sources chosen in Step A3
SOURCE_S02 = f_s02
PC_CONFIG_S01 = dict(CONFIG_S01, kind="incidents")
PC_CONFIG_S02 = dict(CONFIG_S02, kind="risk_taxonomy")

def _jsonable(d):
    return json.loads(json.dumps(d, default=str))

def inspect_s01(inputs, cfg):
    df = load_incidents(inputs[0])
    if df.empty:
        raise ValueError("Empty input table")
    m = resolve_columns(df, INCIDENT_SPEC, "S01 incidents", INCIDENT_REQUIRED)   # stops if schema changed
    return _jsonable({"rows": len(df), "columns": list(df.columns), "mapped": m,
                      "missing": {k: int(df[v].isna().sum()) for k, v in m.items()},
                      "exact_duplicates": int(df.duplicated().sum())})

def prepare_s01(inputs, out, cfg):
    src = inputs[0]
    df = load_incidents(src)
    m = resolve_columns(df, INCIDENT_SPEC, "S01 incidents", INCIDENT_REQUIRED)
    prepared, stats = prepare_table_df(df, m, extra_columns(df, m, INCIDENT_EXTRA_KEYWORDS), cfg)
    if prepared.empty:
        raise ValueError("No prepared table rows")
    target = out / "prepared_incidents.csv"
    prepared.to_csv(target, index=False)
    back = pd.read_csv(target, dtype="string")
    if len(back) != len(prepared) or back.columns.tolist() != prepared.columns.tolist():
        raise ValueError("Exported table does not match prepared data")
    return [target], [{"source": str(src), "output": str(target)}], describe_config(cfg) + f"; stats={stats}"

def inspect_s02(inputs, cfg):
    df = load_risks(inputs[0])
    if df.empty:
        raise ValueError("Empty input table")
    m = resolve_columns(df, RISK_SPEC, "S02 risk repository", RISK_REQUIRED)
    return _jsonable({"rows": len(df), "columns": list(df.columns), "mapped": m,
                      "missing": {k: int(df[v].isna().sum()) for k, v in m.items()},
                      "exact_duplicates": int(df.duplicated().sum())})

def prepare_s02(inputs, out, cfg):
    src = inputs[0]
    df = load_risks(src)
    m = resolve_columns(df, RISK_SPEC, "S02 risk repository", RISK_REQUIRED)
    prepared, stats = prepare_table_df(df, m, extra_columns(df, m, RISK_EXTRA_KEYWORDS), cfg)
    if prepared.empty:
        raise ValueError("No prepared table rows")
    target = out / "prepared_risks.csv"
    prepared.to_csv(target, index=False)
    back = pd.read_csv(target, dtype="string")
    if len(back) != len(prepared) or back.columns.tolist() != prepared.columns.tolist():
        raise ValueError("Exported table does not match prepared data")
    return [target], [{"source": str(src), "output": str(target)}], describe_config(cfg) + f"; stats={stats}"

run_S01 = register_and_run("incidents", SOURCE_S01, inspect_s01, prepare_s01, PC_CONFIG_S01)
run_S02 = register_and_run("risk_taxonomy", SOURCE_S02, inspect_s02, prepare_s02, PC_CONFIG_S02)
print("Cell 8B completed successfully.")

# Part C output should match the Part B output for the same source and configuration
for run, b_path in ((run_S01, prepared_path_s01), (run_S02, prepared_path_s02)):
    same = run["validation"]["output_sha256"][0] == source_hash(b_path)
    print(Path(run["outputs"][0]).name, "identical to Part B output:", same)

SUCCESS: incidents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/incidents/pipeline_manifest.json
SUCCESS: risk_taxonomy | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/risk_taxonomy/pipeline_manifest.json
Cell 8B completed successfully.
prepared_incidents.csv identical to Part B output: True
prepared_risks.csv identical to Part B output: True


In [59]:
# Cell 9 — Repeatability and provenance verification
REPEATABILITY_PASSED = False
assert PIPELINE_RUNS, "No pipeline runs registered. Run at least one selected Cell 8 pathway first."
print("--- Repeatability Verification ---")
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    second = entry["runner"]()
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], f"Source-to-output mapping changed for {kind}"
    print(f"PASS: {kind} — source fingerprints, prepared outputs and mappings match")
print("\nAll selected pipelines passed repeatability smoke checks.")
REPEATABILITY_PASSED = True

--- Repeatability Verification ---
SUCCESS: incidents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/incidents/pipeline_manifest.json
PASS: incidents — source fingerprints, prepared outputs and mappings match
SUCCESS: risk_taxonomy | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/risk_taxonomy/pipeline_manifest.json
PASS: risk_taxonomy — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


### Steps C3 and C4: Generate the documents and package the artefacts

In [60]:
# Cell 10 — Generate Week 3 documents from this run, then package artefacts
import shutil
import zipfile

synthetic = (DATA_MODE == "synthetic")
S01, S02 = EVIDENCE["S01"], EVIDENCE["S02"]
b01, b02 = COMPARISON["S01"], COMPARISON["S02"]
st01, st02 = STATS["S01"], STATS["S02"]

def cs(x):
    return str(x).replace("|", "\\|").replace("\n", " ")

BANNER = ("> **SYNTHETIC PRACTICE DATA.** This run used the supplied practice option, not the real project "
          "sources. It demonstrates the pipeline only and says nothing about the real data or the "
          "application's performance. Re-run with real data before final submission.\n\n") if synthetic else ""

def evidence_text(e):
    mapped = ", ".join(f"{k}→{v}" for k, v in e["mapped_columns"].items())
    reps = "; ".join(f"id {r['record_id']}: {r['label']}" for r in e["representative_records"])
    return cs(f"{e['rows']} rows × {e['column_count']} columns, {e['bytes']:,} bytes. Mapped fields: {mapped}. "
              f"Missing (mapped fields): {e['missing_mapped']}. Exact duplicate rows: {e['exact_duplicate_rows']}; "
              f"duplicate IDs: {e['duplicate_ids']}. SHA-256: {e['sha256'][:16]}… . Representative records: {reps}.")

if synthetic:
    s01_type = "Synthetic practice table standing in for the AIID incidents table. Replaced later by the real AIID snapshot."
    s02_type = "Synthetic practice spreadsheet standing in for the MIT AI Risk Repository. Replaced later by the real file."
    s01_rights = "Synthetic practice dataset created in this notebook; no rights issues; not real incident data."
    s02_rights = s01_rights
    s01_owner, s02_owner = "Synthetic (generated in notebook)", "Synthetic (generated in notebook)"
else:
    s01_type = "Structured table (CSV). AIID incident records: ID, title, description, date. Needed for FR1, FR2, FR4, RAI1."
    s02_type = "Structured table (Excel/CSV). MIT AI Risk Repository risk records with domain taxonomy. Needed for FR3, RAI1."
    s01_rights = ("Public weekly snapshot (JSON/MongoDB/CSV) from incidentdatabase.ai. Terms of use NOT yet verified. "
                  "Raw and prepared data are not committed to GitHub; only manifests and evidence are.")
    s02_rights = ("Public. airisk.mit.edu states that data from the MIT AI Risk Initiative is licensed under CC BY 4.0, so "
                  "attribution is required if reused (Slattery et al., 2024, arXiv:2408.12622). Raw file not committed by default.")
    s01_owner = f"{AIID_SNAPSHOT_DATE}; owner: AI Incident Database (incidentdatabase.ai)"
    s02_owner = f"{RISK_REPO_VERSION}; owner: MIT AI Risk Initiative / MIT FutureTech"

inventory = f"""# Week 3 Data Inventory: AI Incident and Risk Explorer

{BANNER}Generated from the Week 3 notebook run on {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}.

## Requirements these sources serve (Step A1)

| Requirement | Needed information | Source | Why usable / constraint |
|---|---|---|---|
| FR1, FR2 | Incident ID, title, description, date | S01 | Public incident catalogue; reporting bias towards widely covered, English-language cases |
| FR4, RAI1 | Unique, non-missing record IDs and a source link | S01, S02 | Records without an ID cannot be cited, so they are removed during preparation |
| FR3 | Risk domain, sub-domain and description | S02 | Defines the shared risk taxonomy for classification |
| NFR5 | Source fingerprint and snapshot date | S01, S02 manifests | SHA-256 shows when a source changes between refreshes |

## Data inventory (Step A4)

| Source ID | Filename / URL | Type and purpose | Version / date / owner | Access / rights | Inspection evidence |
|---|---|---|---|---|---|
| S01 | `data/raw/{cs(S01['relative_path'])}`; https://incidentdatabase.ai/research/snapshots/ | {cs(s01_type)} | {cs(s01_owner)} | {cs(s01_rights)} | {evidence_text(S01)} |
| S02 | `data/raw/{cs(S02['relative_path'])}`; https://airisk.mit.edu | {cs(s02_type)} | {cs(s02_owner)} | {cs(s02_rights)} | {evidence_text(S02)} |

## Raw versus prepared storage

| Item | Location in the Colab workspace | Committed to GitHub? |
|---|---|---|
| Raw sources S01, S02 | `data/raw/` | No (source terms not confirmed; files are large and temporary) |
| Prepared tables | `data/processed/prepared_incidents.csv`, `prepared_risks.csv` | No by default; only if the source terms allow redistribution |
| Lineage manifest | `data/week3_manifest.json` | Yes |
| Pipeline manifests | `data/pipeline_v1/<run>/pipeline_manifest.json` | Yes |

## Checkpoint A
- **Which requirement needs this data?** FR1/FR2 and FR4 need S01. FR3 needs S02. RAI1 needs stable IDs from both.
- **Loaded sample (not just "file loaded"):** two representative records per source are listed in the inspection evidence above.
- **Rights and replacement:** {"The synthetic data will be replaced by the real AIID snapshot and MIT AI Risk Repository file before the final submission." if synthetic else "Source terms must be confirmed before any raw or prepared data is shared. Until then only manifests and evidence are committed."}
"""

def cmp_table(b):
    rows = ["| Metric | A: as loaded | B: prepared |", "|---|---|---|"]
    for k in b["A_as_loaded"]:
        rows.append(f"| {k.replace('_', ' ')} | {b['A_as_loaded'][k]} | {b['B_prepared'][k]} |")
    return "\n".join(rows)

if synthetic:
    pipeline_banner = BANNER
else:
    pipeline_banner = ""

REPEAT_TEXT = ("Cell 9 reran both pipelines and the source fingerprints, output fingerprints and mappings matched." if globals().get('REPEATABILITY_PASSED') else "Cell 9 has NOT been run yet: run it, then re-run this cell.")

decisions = f"""# Week 3 Pipeline Decisions: AI Incident and Risk Explorer

{pipeline_banner}Generated from the Week 3 notebook run on {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}. Notebook: `notebooks/week3_pipeline.ipynb`. Data type and pathway: structured tables, **Pathway B (CSV/Excel)**, run for two sources.

## 1. Topic 2 requirements and data needs

| Requirement | Data it needs | Source |
|---|---|---|
| FR1, FR2 (answer questions, find similar incidents) | Incident ID, title, description, date | S01 |
| FR3 (classify into risk domains) | Domain, sub-domain, risk description | S02 |
| FR4, RAI1 (cite a verifiable record) | Unique, non-missing IDs; a source link | S01, S02 |
| NFR5 (refresh without manual steps) | Snapshot date; source and output fingerprints | S01, S02 |

The architecture component expanded here is **Data Ingestion & Refresh** feeding the **Incident & Risk Knowledge Store** in `docs/week2_architecture_v1.png`. No embeddings or vector index are built in Week 3.

## 2. Selected sources
- **S01:** `{S01['relative_path']}` (AIID incidents, {S01['rows']} rows).
- **S02:** `{S02['relative_path']}` (MIT AI Risk Repository, {S02['rows']} rows).
- Access, version and rights caveats are in `docs/week3_data_inventory.md`.

## 3. Part B comparison: same input, two configurations

**S01 (AIID incidents)**

{cmp_table(b01)}

**S02 (MIT AI Risk Repository)**

{cmp_table(b02)}

**Selected approach:** configuration B (prepared).
- **Benefit:** it removed {st01['rows_dropped_missing_id']} S01 and {st02['rows_dropped_missing_id']} S02 rows without a usable ID and {st01['exact_duplicates_removed']} S01 and {st02['exact_duplicates_removed']} S02 exact duplicate rows, so every prepared record can be cited and no result is duplicated (FR2, FR4, RAI1).
- **Accepted limitation:** rows with missing descriptions are kept and only flagged ({st01['missing_description_flagged']} in S01, {st02['missing_description_flagged']} in S02), so retrieval quality for those rows is unknown until the Topic 4 quality review. The pipeline also does not reduce the reporting bias already in the source data.

## 4. Chosen preparation configuration
- Standardise column names (`record_id`, `title`, `description`, `date`, and domain fields for S02).
- Trim whitespace; blank text becomes missing; no imputation, no value changes.
- Drop rows without a record ID; remove exact duplicate rows.
- S01 only: build `source_url` from the incident ID (`https://incidentdatabase.ai/cite/<id>`; check that links open).
- Duplicate IDs remaining after cleaning: S01 {st01['duplicate_ids_remaining']}, S02 {st02['duplicate_ids_remaining']} (flagged, not removed).

## 5. ETL / ELT decision
**ETL (transform, then store).** The tables are small, the transformations are light, and no data warehouse is used, so cleaning the data before saving it as CSV keeps the Week 4 retrieval input simple. An ELT design (load raw data into a database, then transform inside it) would add infrastructure that this single-developer project does not need.

## 6. Batch versus streaming
**Batch.** The AIID publishes weekly snapshots, NFR5 asks for a scripted refresh, and real-time monitoring is explicitly out of scope in `docs/week2_requirements.md`. Streaming would add complexity without a requirement.

## 7. Stage contracts and failure points

| Stage | Contract / evidence | Failure response |
|---|---|---|
| Ingest | The chosen source exists and is the only match (`select_source`) | Stop with the list of candidates |
| Inspect | Required columns are found (`resolve_columns`) | Stop and print the available columns |
| Prepare | Prepared table is not empty and has a record ID on every row | Stop; report the transformation failure |
| Validate | Output exists, is non-empty, reads back with the same rows and columns, SHA-256 recorded | Stop for missing or empty output |
| Record lineage | Each source maps to its exact output; checksums re-verified (Cell 6B) | Stop and report the missing mapping |

**Failure point 1: source schema changes (Inspect).** A new snapshot renames or removes a column such as `incident_id`. *Response:* `resolve_columns` stops the run with the list of available columns, and the column spec in the notebook is updated.

**Failure point 2: records without a usable ID (Prepare).** Rows with no ID could not be cited. *Response:* they are dropped and counted (S01 {st01['rows_dropped_missing_id']}, S02 {st02['rows_dropped_missing_id']} in this run), and the count is recorded so a sudden jump is visible after a refresh.

## 8. Output locations
- Part B prepared outputs: `data/processed/prepared_incidents.csv`, `prepared_risks.csv`; evidence in `data/processed/a3_inspection_evidence.json` and `b3_comparison.json`.
- Part B lineage: `data/week3_manifest.json`.
- Part C outputs: `data/pipeline_v1/incidents/` and `data/pipeline_v1/risk_taxonomy/`, each with `pipeline_manifest.json`.
- Repeatability: {REPEAT_TEXT}

## 9. Provenance and access caveats
- SHA-256 proves a file is unchanged. It does not prove permission to share or data quality.
- Source version and rights are recorded as "not verified" in the manifests until confirmed in the inventory.
- Raw data and prepared data are not committed to GitHub unless the source terms allow it.

## 10. Traceability update

| Week 2 item | Week 3 evidence |
|---|---|
| ADR3 (offline snapshot refreshed by script) | The notebook ingests a dated snapshot and records fingerprints (`week3_manifest.json`) |
| FR1, FR2 / NFR2 (retrieval) | `prepared_incidents.csv` is the input for later retrieval; inspection evidence in the inventory |
| FR3 / NFR3 (classification) | `prepared_risks.csv` provides the domain labels |
| FR4 / RAI1 (citations) | Unique, non-missing `record_id` and `source_url` on every prepared record |
| NFR5 (maintainability) | One notebook run reproduces the outputs; Cell 9 checks repeatability |

## 11. Adaptations from the lab starter code
- Pathway B was run twice (S01, S02), so the two Part C runs are named `incidents` and `risk_taxonomy`.
- Added: column-name resolution (the real files' headers may differ), Excel header-row detection, an ID requirement, and a derived `source_url` for S01.
- Cell 1 uses `/content/week3_project` in Colab; a local fallback exists only for testing.
"""

(DOCS_DIR / "week3_data_inventory.md").write_text(inventory, encoding="utf-8")
(DOCS_DIR / "week3_pipeline_decisions.md").write_text(decisions, encoding="utf-8")
print("Written:", DOCS_DIR / "week3_data_inventory.md")
print("Written:", DOCS_DIR / "week3_pipeline_decisions.md")

# Package artefacts in the repository layout
zip_path = BASE_DIR / "week3_github_artifacts.zip"
include = [DOCS_DIR / "week3_data_inventory.md", DOCS_DIR / "week3_pipeline_decisions.md",
           BASE_DIR / "data" / "week3_manifest.json",
           PROCESSED_DIR / "a3_inspection_evidence.json", PROCESSED_DIR / "b3_comparison.json"]
include += sorted(PIPE_DIR.glob("*/pipeline_manifest.json"))
if INCLUDE_PREPARED_DATA_IN_ZIP:
    include += [prepared_path_s01, prepared_path_s02] + sorted(PIPE_DIR.glob("*/prepared_*.csv"))
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in include:
        zf.write(p, p.relative_to(BASE_DIR))
print("\nPackaged for GitHub:", zip_path)
for p in include:
    print(" -", p.relative_to(BASE_DIR))
print("\nPrepared data in zip:", INCLUDE_PREPARED_DATA_IN_ZIP, "(raw data is never included)")

print("\n=== EVIDENCE SUMMARY (check these before submitting) ===")
print("S01 rows in/out:", st01["rows_in"], "->", st01["rows_out"], "| S02 rows in/out:", st02["rows_in"], "->", st02["rows_out"])
print("Repeatability: PASS" if globals().get("REPEATABILITY_PASSED") else "Repeatability: NOT confirmed (run Cell 9)")
if "[fill" in AIID_SNAPSHOT_DATE or "[fill" in RISK_REPO_VERSION:
    print("WARNING: set AIID_SNAPSHOT_DATE and RISK_REPO_VERSION in the config cell, then re-run this cell.")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Not in Colab (or download blocked): use the Files panel to download", zip_path)

Written: /content/week3_project/docs/week3_data_inventory.md
Written: /content/week3_project/docs/week3_pipeline_decisions.md

Packaged for GitHub: /content/week3_project/week3_github_artifacts.zip
 - docs/week3_data_inventory.md
 - docs/week3_pipeline_decisions.md
 - data/week3_manifest.json
 - data/processed/a3_inspection_evidence.json
 - data/processed/b3_comparison.json
 - data/pipeline_v1/incidents/pipeline_manifest.json
 - data/pipeline_v1/risk_taxonomy/pipeline_manifest.json
 - data/processed/prepared_incidents.csv
 - data/processed/prepared_risks.csv
 - data/pipeline_v1/incidents/prepared_incidents.csv
 - data/pipeline_v1/risk_taxonomy/prepared_risks.csv

Prepared data in zip: True (raw data is never included)

=== EVIDENCE SUMMARY (check these before submitting) ===
S01 rows in/out: 1786 -> 1786 | S02 rows in/out: 2574 -> 2574
Repeatability: PASS


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>